# Spark Cluster Playground

This notebook runs locally in Cursor/Jupyter and submits jobs to the Spark cluster from `docker-compose` (`spark://localhost:7077`).

In [7]:
from pathlib import Path
import sys
from pyspark.sql import functions as sf


In [8]:
# Make repo root importable regardless of Jupyter working directory
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from notebooks.spark_session import create_spark_session

spark = create_spark_session('build-candidates-repo')
spark

In [9]:
print('master:', spark.sparkContext.master)
print('app:', spark.sparkContext.appName)


master: spark://spark-master:7077
app: build-candidates-repo


In [10]:
print(spark.sparkContext.master)
print(spark.sparkContext.getConf().get("spark.executor.memory"))
print(spark.sparkContext.getConf().get("spark.executor.cores"))
print(spark.sparkContext.getConf().get("spark.cores.max"))

spark://spark-master:7077
None
None
None


In [11]:
path = "s3a://gba-bronze-zone-prod/gh_events_flat/dt=2026-03-17/hr=22/"
df = spark.read.parquet(path)
df.select("*").distinct().show(5, truncate=False)

+----------+----------+--------------------+----------+----------------------------------------------+---------+-------------------+---------+---------+-------------------------------------+--------------------------------------------------+---------+--------------+----------------+---------------+-------------------+--------+----------+----------+---------+--------------------------+---------------------------------------------------------------------------+
|event_id  |event_type|created_at          |repo_id   |repo_full_name                                |actor_id |actor_login        |org_id   |org_login|org_url                              |org_avatar_url                                    |is_public|payload_action|payload_ref_type|pull_request_id|pull_request_merged|issue_id|comment_id|release_id|member_id|ingestion_ts              |source_file                                                                |
+----------+----------+--------------------+----------+-----------------

In [12]:
df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- actor_id: long (nullable = true)
 |-- actor_login: string (nullable = true)
 |-- org_id: long (nullable = true)
 |-- org_login: string (nullable = true)
 |-- org_url: string (nullable = true)
 |-- org_avatar_url: string (nullable = true)
 |-- is_public: boolean (nullable = true)
 |-- payload_action: string (nullable = true)
 |-- payload_ref_type: string (nullable = true)
 |-- pull_request_id: long (nullable = true)
 |-- pull_request_merged: boolean (nullable = true)
 |-- issue_id: long (nullable = true)
 |-- comment_id: long (nullable = true)
 |-- release_id: long (nullable = true)
 |-- member_id: long (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [ ]:
df.groupBy('org_id', 'org_login').count().show()

In [6]:
df.select('event_type').distinct().show(truncate=False)

+-----------------------------+
|event_type                   |
+-----------------------------+
|PullRequestReviewEvent       |
|PushEvent                    |
|GollumEvent                  |
|ReleaseEvent                 |
|CommitCommentEvent           |
|CreateEvent                  |
|PullRequestReviewCommentEvent|
|IssueCommentEvent            |
|DeleteEvent                  |
|IssuesEvent                  |
|ForkEvent                    |
|PublicEvent                  |
|MemberEvent                  |
|WatchEvent                   |
|PullRequestEvent             |
|DiscussionEvent              |
+-----------------------------+



## Possible aggregations for step 3:

* [x] events total
* [x] unique actors
* [x] push_events, pr_events, issue_events, release_events, ... (distribution by event_type)
* [x] push_ratio (push_events / events_total)
* [x] bot_events and bot_ratio (actor heuristic (actor_login like '%[bot]%' etc.)
* [ ] human_unique_actors
* [ ] source_files_count (?)
* [x] composite_score (0.5*log1p(events_last_24h) + 0.3*log1p(unique_actors) + 0.2*active_hours_count - penalty(bot_ratio)) (?)

### Total number of events per repo (events total):

In [7]:
df.groupBy(df.repo_id).count().show()

[Stage 7:=============================>                             (1 + 1) / 2]

+----------+-----+
|   repo_id|count|
+----------+-----+
| 693456392|    2|
|1175197507|    2|
|1065146145|    1|
|1081562611|    1|
|1178343715|    1|
|1112931994|   23|
|1170595902|    1|
|1170070218|    2|
| 594514315|    1|
| 980149376|    1|
| 567038244|    2|
|1154680779|    3|
|1149594310|    1|
| 629624246|    1|
|1123520631|    3|
|1106256762|   29|
|1178250390|    1|
|1137922759|    1|
| 773615466|    1|
|1177295252|    2|
+----------+-----+
only showing top 20 rows



In [8]:
df.select(df.actor_id).where(df.repo_id== "1106256762").distinct().show()

[Stage 10:=============================>                            (1 + 1) / 2]

+---------+
| actor_id|
+---------+
|246770584|
+---------+



### Unique actors

In [9]:
df.groupBy(df.repo_id).agg(sf.countDistinct("actor_id").alias('actors_count')).show()

[Stage 13:=============================>                            (1 + 1) / 2]

+----------+------------+
|   repo_id|actors_count|
+----------+------------+
|1079469230|           1|
|1176308272|           1|
|1152443889|           1|
|1173271499|           1|
|1054362403|           1|
|1178276808|           1|
| 903336683|           2|
| 402852632|           1|
|  90158362|           1|
|1172386853|           1|
|1143054734|           1|
|1178352742|           1|
|1166143147|           1|
|1178371807|           1|
|1001330787|           1|
| 416882051|           1|
|1177446876|           1|
| 597389977|           1|
|1073624303|           2|
|1154018330|           1|
+----------+------------+
only showing top 20 rows



### Distribution by event_type

In [10]:
df.groupBy(df.repo_id).pivot("event_type").count().na.fill(0).show()

[Stage 27:=============================>                            (1 + 1) / 2]

+----------+------------------+-----------+-----------+---------------+---------+-----------+-----------------+-----------+-----------+-----------+----------------+-----------------------------+----------------------+---------+------------+----------+
|   repo_id|CommitCommentEvent|CreateEvent|DeleteEvent|DiscussionEvent|ForkEvent|GollumEvent|IssueCommentEvent|IssuesEvent|MemberEvent|PublicEvent|PullRequestEvent|PullRequestReviewCommentEvent|PullRequestReviewEvent|PushEvent|ReleaseEvent|WatchEvent|
+----------+------------------+-----------+-----------+---------------+---------+-----------+-----------------+-----------+-----------+-----------+----------------+-----------------------------+----------------------+---------+------------+----------+
| 402144416|                 0|          0|          0|              0|        0|          0|                0|          0|          0|          0|               0|                            0|                     0|        1|           0|    

### Push ratio (push_events / events_total)

In [14]:
repo_metrics = (
      df.groupBy("repo_id", "repo_full_name")
      .agg(
          sf.count("*").alias("events_total"),
          sf.sum(sf.when(sf.col("event_type") == "PushEvent", 1).otherwise(0)).alias("push_events"),
          sf.countDistinct("actor_id").alias("unique_actors"),
          sf.max("created_at").alias("last_event_at"),
      )
      .withColumn(
          "push_ratio",
          sf.when(sf.col("events_total") > 0, sf.col("push_events") / sf.col("events_total"))
           .otherwise(sf.lit(0.0))
      )
  )
repo_metrics.show()

[Stage 33:=============================>                            (1 + 1) / 2]

+-------+--------------------+------------+-----------+-------------+--------------------+------------------+
|repo_id|      repo_full_name|events_total|push_events|unique_actors|       last_event_at|        push_ratio|
+-------+--------------------+------------+-----------+-------------+--------------------+------------------+
|   NULL|                NULL|           6|          0|            4|2026-03-11T00:58:44Z|               0.0|
|  10664|cucumber/cucumber...|           1|          1|            1|2026-03-11T00:07:30Z|               1.0|
| 119165|mzmcbride/databas...|           4|          0|            1|2026-03-11T00:16:21Z|               0.0|
| 285722|       OpenMW/openmw|           1|          1|            1|2026-03-11T00:34:09Z|               1.0|
| 356066|apache/trafficserver|           1|          0|            1|2026-03-11T00:44:13Z|               0.0|
| 394448|    bdarnell/tornado|           1|          0|            1|2026-03-11T00:58:48Z|               0.0|
| 449595|t

### Composite score 

(0.5*log1p(events_last_24h) + 0.3*log1p(unique_actors) + 0.2*active_hours_count - penalty(bot_ratio))

In [22]:
repo_metrics = repo_metrics.withColumn(
      "composite_score",
      0.5 * sf.log1p(sf.col("events_total")) +
      0.3 * sf.col("push_ratio") +
      0.2 * sf.log1p(sf.col("unique_actors"))
)
repo_metrics.select('composite_score').show()

[Stage 45:=============================>                            (1 + 1) / 2]

+-------------------+
|    composite_score|
+-------------------+
| 0.6879355804460439|
| 0.7852030263919617|
| 0.9190286020676768|
| 1.5285514187210012|
| 0.7852030263919617|
|0.48520302639196167|
| 1.5392511225412513|
| 0.7852030263919617|
|0.48520302639196167|
| 0.7852030263919617|
|0.48520302639196167|
| 1.0128696382935671|
| 0.7852030263919617|
| 0.7852030263919617|
| 0.7852030263919617|
| 0.7852030263919617|
| 0.7852030263919617|
| 0.8317766166719344|
| 0.7852030263919617|
| 1.0690286020676767|
+-------------------+
only showing top 20 rows



### bot_events and bot_ratio (actor heuristic (actor_login like '%[bot]%' etc.)

In [38]:
# df.select(df.actor_id, df.actor_login).filter(df.actor_login.ilike('%[bot]')).show()
repo_metrics = df.groupBy("repo_id", "repo_full_name").agg(
    # sf.ilike("actor_login", "%[bot]").alias('is_bot')
    sf.count("*").alias("events_total"),
    sf.sum(
        sf.when(
            sf.lower(sf.col("actor_login")).ilike("%[bot]"),
            1
        ).otherwise(0)
    ).alias("bot_events")
)
repo_metrics.show()

[Stage 60:=============================>                            (1 + 1) / 2]

+----------+--------------------+------------+----------+
|   repo_id|      repo_full_name|events_total|bot_events|
+----------+--------------------+------------+----------+
| 902664388|UltracoldAtomsLab...|           3|         0|
|1172182111|christalpro/Data-...|           1|         0|
|1123517130|brentiswong/ocean...|           3|         3|
|1174352733|    CosimoZi/writing|           4|         0|
|1178221690|Mclovin0213/basie-64|           1|         0|
|1141615754|KanuckLemonFTW/Di...|           1|         1|
|1026676997|yasminSarbaoui93/...|           1|         0|
|1178336127|DioXy8/oniry.gith...|           1|         0|
|1178343163|LOLOPOLLOWEY/asis...|           1|         0|
| 992927189|QCADevProd/qdo-ca...|          13|         9|
|1175425083|myashiferaw-png/S...|           1|         0|
|1125551250|NurraHealth/ccda-...|           2|         0|
|1144519171|blowell19/ds334_blog|           1|         0|
| 889693983|gmlewis/moonbit-sha1|           1|         0|
| 656400292|  

In [39]:
repo_metrics = repo_metrics.withColumn(
      "bot_ratio",
      sf.when(sf.col("events_total") > 0, sf.col("bot_events") / sf.col("events_total"))
       .otherwise(sf.lit(0.0))
  )
repo_metrics.show()

[Stage 63:=============================>                            (1 + 1) / 2]

+----------+--------------------+------------+----------+------------------+
|   repo_id|      repo_full_name|events_total|bot_events|         bot_ratio|
+----------+--------------------+------------+----------+------------------+
| 902664388|UltracoldAtomsLab...|           3|         0|               0.0|
|1172182111|christalpro/Data-...|           1|         0|               0.0|
|1123517130|brentiswong/ocean...|           3|         3|               1.0|
|1174352733|    CosimoZi/writing|           4|         0|               0.0|
|1178221690|Mclovin0213/basie-64|           1|         0|               0.0|
|1141615754|KanuckLemonFTW/Di...|           1|         1|               1.0|
|1026676997|yasminSarbaoui93/...|           1|         0|               0.0|
|1178336127|DioXy8/oniry.gith...|           1|         0|               0.0|
|1178343163|LOLOPOLLOWEY/asis...|           1|         0|               0.0|
| 992927189|QCADevProd/qdo-ca...|          13|         9|0.6923076923076923|

#### Number of all events for particular repo

In [6]:
df.select(df.repo_id).distinct().show()

+----------+
|   repo_id|
+----------+
| 693456392|
|1175197507|
|1065146145|
|1081562611|
|1178343715|
|1112931994|
|1170595902|
|1170070218|
| 594514315|
| 980149376|
| 567038244|
|1154680779|
|1149594310|
| 629624246|
|1123520631|
|1106256762|
|1178250390|
|1137922759|
| 773615466|
|1177295252|
+----------+
only showing top 20 rows



In [18]:
df.groupBy(df.repo_id, df.repo_full_name).agg(
    sf.count("*").alias('total_events'),
    sf.count("actor_id").alias('actors_count')
).show()

[Stage 37:=============================>                            (1 + 1) / 2]

+----------+--------------------+------------+------------+
|   repo_id|      repo_full_name|total_events|actors_count|
+----------+--------------------+------------+------------+
| 902664388|UltracoldAtomsLab...|           3|           3|
|1172182111|christalpro/Data-...|           1|           1|
|1123517130|brentiswong/ocean...|           3|           3|
|1174352733|    CosimoZi/writing|           4|           4|
|1178221690|Mclovin0213/basie-64|           1|           1|
|1141615754|KanuckLemonFTW/Di...|           1|           1|
|1026676997|yasminSarbaoui93/...|           1|           1|
|1178336127|DioXy8/oniry.gith...|           1|           1|
|1178343163|LOLOPOLLOWEY/asis...|           1|           1|
| 992927189|QCADevProd/qdo-ca...|          13|          13|
|1175425083|myashiferaw-png/S...|           1|           1|
|1125551250|NurraHealth/ccda-...|           2|           2|
|1144519171|blowell19/ds334_blog|           1|           1|
| 889693983|gmlewis/moonbit-sha1|       

In [7]:
event_types = df.select(df.event_type).where(df.repo_id == "693456392").distinct()
events_list = event_types.collect()
# df.select(df.event_type).where(df.repo_id == "693456392").distinct().show()

In [17]:
for event in events_list:
    df.where((df.repo_id == "693456392") & (df.event_type == event.event_type)).groupBy(df.repo_id).count().show()

[Stage 25:=============================>                            (1 + 1) / 2]

+---------+-----+
|  repo_id|count|
+---------+-----+
|693456392|    2|
+---------+-----+



In [ ]:
df.where((df.repo_id == "44838949") & (df.event_type == 'PullRequestReviewCommentEvent')).groupBy(df.repo_id).count().show()

+--------+-----+
| repo_id|count|
+--------+-----+
|44838949|    3|
+--------+-----+



In [11]:
df.select('actor_id').distinct().where(df.repo_id == "44838949").show()

+--------+
|actor_id|
+--------+
|   66486|
|   21240|
|14333906|
+--------+



### Global metrics

In [13]:
import re

def camel_to_snake(value: str) -> str:
  return re.sub(r"(?<!^)(?=[A-Z])", "_", value).lower()

In [14]:
event_types_list = [item.event_type for item in df.select('event_type').distinct().collect()]
event_types_list

['PullRequestReviewEvent',
 'PushEvent',
 'GollumEvent',
 'ReleaseEvent',
 'CommitCommentEvent',
 'CreateEvent',
 'PullRequestReviewCommentEvent',
 'IssueCommentEvent',
 'DeleteEvent',
 'IssuesEvent',
 'ForkEvent',
 'PublicEvent',
 'MemberEvent',
 'WatchEvent',
 'PullRequestEvent',
 'DiscussionEvent']

In [15]:
dt = "2026-03-17"
hr = "22"
event_columns = []
ration_columns = {}
for raw_name in event_types_list:
    alias_name = f"{camel_to_snake(raw_name)}s"
    column = sf.sum(sf.when(sf.col("event_type") == raw_name, 1).otherwise(0)).alias(alias_name)
    ration_columns[f"{alias_name}_ratio"] = sf.when(
        sf.col("total_events") > 0, sf.col(alias_name) / sf.col("total_events")
    ).otherwise(sf.lit(0.0))
    event_columns.append(column)

In [16]:
repo_metrics = df.groupBy("repo_id", "repo_full_name").agg(
    sf.count("*").alias("total_events"),
    sf.countDistinct("actor_id").alias("unique_actors"),
    sf.max("created_at").alias("last_event_at"),
    sf.sum(
        sf.when(
            sf.lower(sf.col("actor_login")).ilike("%[bot]"),
            1
        ).otherwise(0)
    ).alias("bot_events"),
    sf.countDistinct("issue_id").alias("issues_count"),
    sf.countDistinct("comment_id").alias("comments_count"),
    sf.countDistinct("release_id").alias("releases_count"),
    sf.countDistinct("pull_request_merged").alias("merged_pr_count"),
    *event_columns
)

for key, col in ration_columns.items():    
    repo_metrics = repo_metrics.withColumn(key, col)

repo_metrics = (
    repo_metrics
        .withColumn(
            "composite_score",
              0.5 * sf.log1p(sf.col("total_events")) +
              0.3 * sf.col("push_events_ratio") +
              0.2 * sf.log1p(sf.col("unique_actors"))
        )
        .withColumn(
              "bot_ratio",
              sf.when(sf.col("total_events") > 0, sf.col("bot_events") / sf.col("total_events"))
               .otherwise(sf.lit(0.0))
        )
        .withColumn("dt", sf.lit(dt).cast("date"))
        .withColumn("hr", sf.lit(hr).cast("int"))
        .withColumn(
          "last_event_at",
          sf.to_timestamp("last_event_at", "yyyy-MM-dd'T'HH:mm:ssX")
        )
)

In [17]:
repo_metrics.printSchema()

root
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- total_events: long (nullable = false)
 |-- unique_actors: long (nullable = false)
 |-- last_event_at: string (nullable = true)
 |-- bot_events: long (nullable = true)
 |-- issues_count: long (nullable = false)
 |-- comments_count: long (nullable = false)
 |-- releases_count: long (nullable = false)
 |-- merged_pr_count: long (nullable = false)
 |-- pull_request_review_events: long (nullable = true)
 |-- push_events: long (nullable = true)
 |-- gollum_events: long (nullable = true)
 |-- release_events: long (nullable = true)
 |-- commit_comment_events: long (nullable = true)
 |-- create_events: long (nullable = true)
 |-- pull_request_review_comment_events: long (nullable = true)
 |-- issue_comment_events: long (nullable = true)
 |-- delete_events: long (nullable = true)
 |-- issues_events: long (nullable = true)
 |-- fork_events: long (nullable = true)
 |-- public_events: long (nullable = true)

In [22]:
repo_metrics.select(
    "repo_id",
    "repo_full_name",
    "dt",
    "hr",
    "issues_count",
    "comments_count",
    "releases_count",
    "merged_pr_count"
).orderBy(sf.desc("comments_count")).show()

[Stage 31:=============================>                            (1 + 1) / 2]

+----------+--------------------+----------+---+------------+--------------+--------------+---------------+
|   repo_id|      repo_full_name|        dt| hr|issues_count|comments_count|releases_count|merged_pr_count|
+----------+--------------------+----------+---+------------+--------------+--------------+---------------+
|1129449105|Cyber-Security-On...|2026-03-17| 22|          80|           128|             0|              0|
|1103012935|   openclaw/openclaw|2026-03-17| 22|          29|            47|             0|              0|
|1167463379|ironkid90/codex-o...|2026-03-17| 22|           2|            35|             0|              0|
|1176127726|daler91/Hyrox-Com...|2026-03-17| 22|          16|            26|             0|              0|
| 677695414|matthewhand/open-...|2026-03-17| 22|          15|            26|             0|              0|
| 998139995|issdandavis/aethr...|2026-03-17| 22|          22|            22|             0|              0|
|1110902418|   hbmartin/svrn

In [23]:
 spark.stop()